# 工程35｜実際のPyBulletロボット動作動画を作る

作成：2026/10/20 05:36 JST

**目的：説明動画ではなく、ロボットがりんごを掴み・持ち上げ・横移動し・受け台に置く動作をそのままMP4に収録する。** 既存の制御や安全確認条件は変更しない。

## 1｜工程35 ZIPをアップロード

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
archive=Path('PAI_stage35_actual_motion_20261020.zip')
assert archive.name in uploaded, '工程35のZIPを選択してください'
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    z.extractall('/content')
root=Path('/content/PAI_stage35_actual_motion_20261020')
assert (root/'run_stage35_motion.py').is_file()
print('READY',root)

## 2｜PyBulletとAIライブラリ

In [ ]:
%cd /content/PAI_stage35_actual_motion_20261020
%pip -q install -r requirements.txt

## 3｜コードとAIモデルの準備（運転前に一度だけ学習）

In [ ]:
!python -m py_compile run_stage35_motion.py concat_stage35_videos.py
!python -u run_stage35_motion.py --train-only

## 4｜まず実物理4件を撮影する（各等級1件）

In [ ]:
!python -u run_stage35_motion.py --cycles 1 --output stage35_preview_4
!python -u concat_stage35_videos.py --source stage35_preview_4 --output /content/stage35_preview_4.mp4
from IPython.display import Video,display
display(Video('/content/stage35_preview_4.mp4', embed=True))

**実際のロボットが掴み、運び、離して中央へ戻っているかを上の動画で確認する。** 画角が悪ければ、続く24件は実行しない。

## 5｜動画をダウンロード（4件の映像プレビュー）

In [ ]:
files.download('/content/stage35_preview_4.mp4')

## 6｜画角・動作を確認後、約1分強の本収録（4等級×6周＝24試行）

In [ ]:
!python -u run_stage35_motion.py --cycles 6 --output stage35_full_24
!python -u concat_stage35_videos.py --source stage35_full_24 --output /content/stage35_pybullet_motion_24.mp4
from IPython.display import Video,display
display(Video('/content/stage35_pybullet_motion_24.mp4', embed=False))

## 7｜シミュレーション動画と実験記録を保存

In [ ]:
from pathlib import Path
import zipfile
out=Path('/content/stage35_pybullet_motion_package.zip')
root=Path('/content/PAI_stage35_actual_motion_20261020')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    z.write('/content/stage35_pybullet_motion_24.mp4','stage35_pybullet_motion_24.mp4')
    for f in (root/'stage35_full_24').rglob('*'):
        if f.is_file() and ('physics_trial.mp4' not in f.name) and not ('_engine/' in str(f)):
            if '_engine' in f.parts or 'engine' in f.parts:continue
            z.write(f,str(f.relative_to(root)))
print('READY',out,out.stat().st_size)
files.download(str(out))